# LaStBeRu × Rubin reference catalog

This notebook runs the large-catalog workflow in separate stages:

1. **TAP coverage:** counts individual images and summarizes seeing, limiting magnitude, exposure time, and epochs by band.
2. **Coadd properties:** optionally samples coadd quality maps only for targets with individual-image coverage.
3. **Cutouts:** optionally loads full coadds only for the prioritized targets.

Run the first stage over all targets before enabling the more expensive stages.

**Criterio geométrico actual:** `detector_search_radius_deg=0.35` es solo un radio de preselección para traer detectores candidatos. El matching final considera que el punto RA/DEC del target esté dentro del cuadrilátero del detector; no considera todavía un disco de 10 arcsec alrededor del target.


In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from target_selection.reference_catalog import make_science_rgb

from target_selection import (
    build_cutout_pdf,
    load_reference_catalog_config,
    run_reference_coverage_catalog,
    enrich_reference_catalog_coadds,
    generate_reference_catalog_cutouts,
)

_repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (parent for parent in _repo_candidates if (parent / "configs" / "lastberu_dp2.toml").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the target_selection repository or its notebooks/ directory.")

CONFIG_PATH = REPO_ROOT / "configs" / "lastberu_dp2.toml"
TARGET_LIMIT = None  # Set None for all targets.
RUN_COVERAGE = False  # reutiliza coverage_catalog.csv si ya existe
RUN_COADD_DETECTION = True
RUN_COADD_PROPERTIES = True
RUN_CUTOUTS = True  # Generate the configured A/B cutouts
REGENERATE_CUTOUTS = False  # Set False after the new layout has been generated.
MAX_CUTOUTS = 100_000  # procesa todos los candidatos A/B

config = load_reference_catalog_config(CONFIG_PATH)
run_name = config.name if TARGET_LIMIT is None else f"{config.name}_preview_{TARGET_LIMIT}"
config = replace(
    config,
    name=run_name,
    target_limit=TARGET_LIMIT,
    max_cutouts=MAX_CUTOUTS,
    save_cutout_fits=True,
    input_path=str(REPO_ROOT / config.input_path),
    output_dir=str(REPO_ROOT / config.output_dir),
    cache_dir=str(REPO_ROOT / config.cache_dir),
)
output_directory = Path(config.output_dir) / config.name
FITS_OUTPUT_DIR = REPO_ROOT / "delete_sundays" / "lastberu_cutouts" / "lastberu_completo" / "fits"
CUTOUT_OUTPUT_DIR = REPO_ROOT / "delete_sundays" / "lastberu_cutouts" / "lastberu_completo"
RGB_OUTPUT_DIR = CUTOUT_OUTPUT_DIR / "rgb"
config

## PDF de cutouts

Una página por target, con un panel para cada FITS de banda disponible. La lógica de lectura y visualización está en `target_selection.collection_access`.

In [ ]:
pdf_path = build_cutout_pdf(
    catalog, fits_dir=FITS_OUTPUT_DIR,
    output_path=CUTOUT_OUTPUT_DIR / "lastberu_cutouts.pdf",
    bands=bands, verbose=True,
)
print(f"PDF guardado: {pdf_path}")

## 1. TAP coverage and individual-image summaries

This is the fast first pass. It does not initialize Butler and does not read coadd maps.

**Criterio geométrico actual:** `detector_search_radius_deg=0.35` es solo un radio de preselección para traer detectores candidatos. El matching final considera que el punto RA/DEC del target esté dentro del cuadrilátero del detector; no considera todavía un disco de 10 arcsec alrededor del target.


In [ ]:
coverage_path = output_directory / "coverage_catalog.csv"
if RUN_COVERAGE:
    coverage_catalog = run_reference_coverage_catalog(config)
elif coverage_path.exists():
    coverage_catalog = pd.read_csv(coverage_path)
else:
    raise FileNotFoundError(f"No coverage catalog at {coverage_path}. Set RUN_COVERAGE=True first.")

print(f"Coverage catalog: {len(coverage_catalog):,} targets")
coverage_catalog.head()

## Exportación de visitas individuales

La consulta de cobertura se ejecuta una sola vez. `coverage_catalog` contiene una fila por target con los conteos y estadísticas agregadas por banda; los siguientes productos se generan reutilizando ese resultado, sin volver a consultar TAP.

In [ ]:
# Comprobación rápida del catálogo de cobertura ya calculado.
coverage_catalog = coverage_catalog.copy()
coverage_catalog["selection_grade"] = coverage_catalog["selection_grade"].fillna("Unknown").astype(str)
coverage_catalog["n_images_total"] = pd.to_numeric(coverage_catalog["n_images_total"], errors="coerce").fillna(0).astype(int)
coverage_catalog["n_filters_with_images"] = coverage_catalog[[f"n_images_{b}" for b in config.bands]].gt(0).sum(axis=1).astype(int)
print(f"Targets en el catálogo: {len(coverage_catalog):,}")
print(f"Targets with at least one visit: {(coverage_catalog.n_images_total > 0).sum():,}")
print(coverage_catalog["selection_grade"].value_counts(dropna=False).sort_index())


## Detección de coadds

Esta etapa consulta Butler para saber si existe un coadd en la posición de cada target. Si `RUN_COADD_PROPERTIES=True`, también muestrea los mapas de profundidad y PSF; esos valores se usan en los resúmenes y distribuciones.

In [ ]:
reference_catalog_path = output_directory / "reference_catalog.csv"
if RUN_COADD_DETECTION:
    catalog = enrich_reference_catalog_coadds(config, coverage_catalog, sample_properties=RUN_COADD_PROPERTIES)
elif reference_catalog_path.exists():
    catalog = pd.read_csv(reference_catalog_path)
else:
    raise FileNotFoundError(f"No existe {reference_catalog_path}; ejecutá RUN_COADD_DETECTION=True.")

# Derivar cobertura de coadd by band incluso cuando solo se ejecutó la detección directa.
for band in config.bands:
    col = f"has_coadd_{band}"
    if col not in catalog:
        catalog[col] = catalog.get("coadd_bands", "").fillna("").astype(str).map(
            lambda value, b=band: b in {x.strip() for x in value.split(",") if x.strip()}
        )

catalog["has_coadd"] = catalog["has_coadd"].fillna(False).astype(bool)
catalog["selection_grade"] = catalog["selection_grade"].fillna("Unknown").astype(str)
reference_catalog_path = output_directory / "reference_catalog.csv"
catalog.to_csv(reference_catalog_path, index=False)
print(f"Catálogo enriquecido: {reference_catalog_path}")
print(f"Targets with coadd coverages: {catalog.has_coadd.sum():,}")


## Tablas de cobertura y calidad

Se generan dos tablas: una por banda y otra por grade. Los conteos se calculan sobre todos los targets del catálogo; por eso los targets sin visitas permanecen en el denominador. Las columnas de profundidad y seeing corresponden a las estadísticas de las visitas individuales; las columnas `coadd_*` corresponden a los mapas de propiedades de coadd si fueron muestreados.

In [ ]:
import numpy as np

out = output_directory
bands = tuple(config.bands)
cat = catalog.copy()
cat["any_visit"] = cat["n_images_total"].gt(0)
cat["n_filters"] = cat[[f"n_images_{b}" for b in bands]].gt(0).sum(axis=1)

rows = []
for b in bands:
    n = pd.to_numeric(cat[f"n_images_{b}"], errors="coerce").fillna(0)
    seeing = pd.to_numeric(cat.get(f"visit_seeing_mean_arcsec_{b}"), errors="coerce")
    depth = pd.to_numeric(cat.get(f"visit_maglim_mean_{b}"), errors="coerce")
    coadd_depth = pd.to_numeric(cat.get(f"coadd_maglim_{b}"), errors="coerce")
    coadd_psf = pd.to_numeric(cat.get(f"coadd_psf_fwhm_arcsec_{b}"), errors="coerce")
    rows.append({
        "band": b, "targets_total": len(cat),
        "targets_with_visit": int(n.gt(0).sum()),
        "individual_images": int(n.sum()),
        "mean_images_per_target": float(n.mean()),
        "median_images_per_target_covered": float(n[n.gt(0)].median()) if n.gt(0).any() else np.nan,
        "mean_visit_seeing_arcsec": float(seeing.mean()),
        "mean_visit_maglim": float(depth.mean()),
        "targets_with_coadd": int(cat[f"has_coadd_{b}"].sum()),
        "mean_coadd_maglim": float(coadd_depth.mean()),
        "mean_coadd_psf_fwhm_arcsec": float(coadd_psf.mean()),
    })
summary_by_band = pd.DataFrame(rows)
total = {"band": "total", "targets_with_visit": int(cat.any_visit.sum()), "individual_images": int(cat.n_images_total.sum()), "mean_images_per_target": float(cat.n_images_total.mean()), "median_images_per_target_covered": float(cat.loc[cat.n_images_total.gt(0), "n_images_total"].median()), "mean_visit_seeing_arcsec": float(cat[[f"visit_seeing_mean_arcsec_{b}" for b in bands]].mean(axis=1).mean()), "mean_visit_maglim": float(cat[[f"visit_maglim_mean_{b}" for b in bands]].mean(axis=1).mean()), "targets_with_coadd": int(cat.has_coadd.sum()), "mean_coadd_maglim": float(cat[[f"coadd_maglim_{b}" for b in bands]].mean(axis=1).mean()), "mean_coadd_psf_fwhm_arcsec": float(cat[[f"coadd_psf_fwhm_arcsec_{b}" for b in bands]].mean(axis=1).mean())}
summary_by_band = pd.concat([summary_by_band, pd.DataFrame([total])], ignore_index=True)
summary_by_band.to_csv(out / "summary_by_band.csv", index=False)

grade_rows = []
for grade, group in cat.groupby("selection_grade", dropna=False):
    row = {
        "selection_grade": grade, "targets_total": len(group),
        "targets_with_any_visit": int(group.any_visit.sum()),
        "targets_with_coadd": int(group.has_coadd.sum()),
        "mean_filters_with_visit": float(group.n_filters.mean()),
        "mean_individual_images": float(group.n_images_total.mean()),
    }
    for b in bands:
        row[f"targets_with_visit_{b}"] = int(pd.to_numeric(group[f"n_images_{b}"], errors="coerce").fillna(0).gt(0).sum())
        row[f"targets_with_coadd_{b}"] = int(group[f"has_coadd_{b}"].sum())
    grade_rows.append(row)
summary_by_grade = pd.DataFrame(grade_rows).sort_values("selection_grade")
summary_by_grade.to_csv(out / "summary_by_grade.csv", index=False)

def save_table_png(frame, path, title, labels, widths, figsize):
    shown = frame.copy()
    for col in shown.columns[1:]:
        shown[col] = shown[col].map(lambda x: "" if pd.isna(x) else f"{x:,.2f}" if isinstance(x, (float, np.floating)) else f"{int(x):,}")
    fig, ax = plt.subplots(figsize=figsize, dpi=220); ax.axis("off")
    widths = np.asarray(widths, dtype=float); widths /= widths.sum()
    table = ax.table(cellText=shown.values, colLabels=labels, colWidths=widths, cellLoc="center", loc="center", bbox=[0, 0, 1, .78])
    table.auto_set_font_size(False); table.set_fontsize(3.8); table.scale(1, .78)
    for (r, c), cell in table.get_celld().items():
        cell.set_facecolor("white"); cell.set_edgecolor("black"); cell.set_linewidth(0); cell.PAD = .002; cell.set_text_props(fontfamily="serif", fontsize=4.0)
        if r == 0: cell.visible_edges = "TB"; cell.set_linewidth(.8); cell.set_text_props(weight="bold", fontfamily="serif"); cell.set_height(cell.get_height() * 1.25)
        elif r == len(shown): cell.visible_edges = "B"; cell.set_linewidth(.8)
        if c == 0 and r > 0: cell.set_text_props(weight="bold", fontfamily="serif")
    ax.set_title(title, fontfamily="serif", fontsize=6, fontweight="bold", pad=-5)
    fig.savefig(path, dpi=220, bbox_inches="tight", pad_inches=.02); plt.close(fig)

band_display = summary_by_band[["band", "targets_with_visit", "individual_images", "mean_images_per_target", "targets_with_coadd"]].rename(columns={"band": "Band", "targets_with_visit": "Targets with visit", "individual_images": "Individual images", "mean_images_per_target": "Average images per target", "targets_with_coadd": "Targets with coadd coverage"})
save_table_png(band_display, out / "summary_by_band.png", f"LaStBeRu × DP2 — {len(cat):,} total targets", ["Band", "Targets with\nvisit", "Individual\nimages", "Average images\nper target", "Coadd coverage\n(targets)"], [.2] * 5, (3.5, 1.5))

grade_display = summary_by_grade[["selection_grade", "targets_total", "targets_with_any_visit", "targets_with_coadd"] + [f"targets_with_visit_{b}" for b in bands]].rename(columns={"selection_grade": "Category", "targets_total": "Total targets", "targets_with_any_visit": "Targets with visit", "targets_with_coadd": "Targets with coadd coverage", **{f"targets_with_visit_{b}": b for b in bands}})
save_table_png(grade_display, out / "summary_by_grade.png", "LaStBeRu × DP2 — categories", ["Category", "Systems\n(total)", "With\nvisits", "Coadd\ncoverage"] + [f"Visits\n{b}" for b in bands], [.12, .12, .15, .16] + [.055] * len(bands), (4.8, max(1.15, .17 * len(grade_display))))

print(f"Saved: {out / 'summary_by_band.csv'} y {out / 'summary_by_grade.csv'}")
display(summary_by_band.round(2))
display(summary_by_grade.round(2))


## Distribuciones

Estos histogramas muestran la distribución por target de la cantidad de visitas, cantidad de filtros con visitas, seeing promedio y profundidad promedio. Las líneas verticales separan visualmente A/B del resto cuando hay grades disponibles.

In [ ]:
from matplotlib.ticker import MaxNLocator

colors = dict(zip(bands, plt.get_cmap("tab10").colors[:len(bands)]))
def numeric_series(frame, column):
    return pd.to_numeric(frame[column], errors="coerce") if column in frame else pd.Series(np.nan, index=frame.index)
def overlay_hist(ax, columns, title, xlabel):
    available = [numeric_series(cat, col).dropna() for col in columns]
    available = [x for x in available if len(x)]
    if not available: return
    edges = np.histogram_bin_edges(pd.concat(available), bins=25)
    for band, values in zip(bands, [numeric_series(cat, col).dropna() for col in columns]):
        if len(values): ax.hist(values, bins=edges, color=colors[band], edgecolor=colors[band], alpha=.22, linewidth=1.4, label=band)
    ax.set(title=title, xlabel=xlabel, ylabel="Targets"); ax.legend(title="Band", ncol=3, fontsize=8); ax.grid(alpha=.25); ax.xaxis.set_major_locator(MaxNLocator(nbins=8))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
visit_arrays = [numeric_series(cat, f"n_images_{b}").fillna(0) for b in bands]
coadd_flags = pd.DataFrame({b: (cat[f"has_coadd_{b}"].fillna(False).astype(bool) if f"has_coadd_{b}" in cat else cat.get("coadd_bands", pd.Series("", index=cat.index)).fillna("").astype(str).map(lambda x, band=b: band in {v.strip() for v in x.split(",") if v.strip()})) for b in bands}, index=cat.index)
max_visits = max(int(x.max()) for x in visit_arrays) if visit_arrays else 1
visit_bins = np.arange(-.5, max_visits + 1.5, 1)
axes[0, 0].hist(visit_arrays, bins=visit_bins, stacked=True, color=[colors[b] for b in bands], label=bands, edgecolor="none")
axes[0, 0].set(title="Individual visits per target", xlabel="Number of visits", ylabel="Targets"); axes[0, 0].legend(title="Band", ncol=3, fontsize=8); axes[0, 0].grid(alpha=.25); axes[0, 0].xaxis.set_major_locator(MaxNLocator(nbins=8, integer=True))
x = np.arange(len(bands)); width = .38
visit_targets = [int(n.gt(0).sum()) for n in visit_arrays]
coadd_targets = [int(coadd_flags[b].sum()) for b in bands]
axes[0, 1].bar(x-width/2, visit_targets, width, label="Visits", color="tab:blue"); axes[0, 1].bar(x+width/2, coadd_targets, width, label="Coadds", color="tab:orange")
axes[0, 1].set(xticks=x, xticklabels=bands, title="Visits and coadd coverage by band", xlabel="Band", ylabel="Targets"); axes[0, 1].legend(); axes[0, 1].grid(alpha=.25, axis="y")
n_visit_filters = pd.concat(visit_arrays, axis=1).gt(0).sum(axis=1); n_coadd_filters = coadd_flags.sum(axis=1)
k = np.arange(len(bands)+1); bins = np.arange(-.5, len(bands)+1.5, 1)
axes[0, 2].hist([n_visit_filters, n_coadd_filters], bins=bins, label=["Visits", "Coadds"], color=["tab:blue", "tab:orange"], alpha=.65, rwidth=.85)
axes[0, 2].set(xticks=k, title="Number of bands per target", xlabel="Number of bands", ylabel="Targets"); axes[0, 2].legend(); axes[0, 2].grid(alpha=.25); axes[0, 2].xaxis.set_major_locator(MaxNLocator(integer=True))
overlay_hist(axes[1, 0], [f"coadd_maglim_{b}" for b in bands], "Coadd depth by band", "Limiting magnitude")
overlay_hist(axes[1, 1], [f"coadd_psf_fwhm_arcsec_{b}" for b in bands], "Coadd PSF by band", "FWHM [arcsec]")
overlay_hist(axes[1, 2], [f"visit_seeing_mean_arcsec_{b}" for b in bands], "Visit seeing by band", "Seeing [arcsec]")
fig.suptitle("LaStBeRu × DP2 — distributions", fontsize=13); fig.tight_layout(); fig.savefig(out / "distributions_dp2.png", dpi=200); plt.show(); plt.close(fig)
print(f"Saved: {out / 'distributions_dp2.png'}")
# Band-by-category counts (grades).
cat["category"] = cat["selection_grade"].fillna("Unknown").astype(str)
category_order = sorted(cat["category"].unique())
category_colors = dict(zip(category_order, plt.get_cmap("tab20").colors))
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(len(bands)); width = .8 / max(len(category_order), 1)
panel_totals = ([numeric_series(cat, f"n_images_{b}").fillna(0).sum() for b in bands], [numeric_series(cat, f"n_images_{b}").fillna(0).gt(0).sum() for b in bands], [coadd_flags[b].sum() for b in bands])
for index, category in enumerate(category_order):
    group = cat.loc[cat["category"].eq(category)]
    offset = (index - (len(category_order)-1)/2) * width
    images = [numeric_series(group, f"n_images_{b}").fillna(0).sum() for b in bands]
    targets_visit = [numeric_series(group, f"n_images_{b}").fillna(0).gt(0).sum() for b in bands]
    targets_coadd = [coadd_flags.loc[group.index, b].sum() for b in bands]
    axes[0].bar(x + offset, images, width, color=category_colors[category], label=category)
    axes[1].bar(x + offset, targets_visit, width, color=category_colors[category], label=category)
    axes[2].bar(x + offset, targets_coadd, width, color=category_colors[category], label=category)
for axis, title, ylabel, totals in zip(axes, ("Individual images by grade", "Targets with individual images by grade", "Targets with coadd coverages by grade"), ("Images", "Targets", "Targets"), panel_totals):
    axis.set(xticks=x, xticklabels=[f"{b}\nN={int(total):,}" for b, total in zip(bands, totals)], title=title, xlabel="Band", ylabel=ylabel); axis.grid(alpha=.25, axis="y")
axes[0].legend(title="Grade", fontsize=8, ncol=2)
fig.suptitle("LaStBeRu × DP2 — band counts by grade", fontsize=13); fig.tight_layout(); fig.savefig(out / "distributions_by_category.png", dpi=200); plt.close(fig)


## Mapas de cobertura en el cielo

Cada punto representa un target y el color indica la cantidad de visitas individuales. Se guarda un mapa total y uno para cada banda.

In [ ]:
def save_visit_map(frame, count_col, filename, title):
    values = pd.to_numeric(frame[count_col], errors="coerce").fillna(0)
    fig, ax = plt.subplots(figsize=(10, 5), dpi=180)
    if values.max() > 0:
        sc = ax.scatter(frame["ra_deg"] % 360, frame["dec_deg"], c=values, s=5, cmap="viridis",
                        norm=plt.matplotlib.colors.LogNorm(vmin=1, vmax=values.max()), alpha=.8, linewidths=0)
    else:
        sc = ax.scatter(frame["ra_deg"] % 360, frame["dec_deg"], c=values, s=5, cmap="viridis", alpha=.8, linewidths=0)
    ax.set(xlabel="RA [deg]", ylabel="Dec [deg]", title=title)
    ax.invert_xaxis(); ax.grid(alpha=.2); fig.colorbar(sc, ax=ax, label="Individual visits")
    fig.tight_layout(); fig.savefig(out / filename, bbox_inches="tight"); plt.close(fig)

save_visit_map(cat, "n_images_total", "sky_map_visits_total.png", "LaStBeRu × DP2 — individual visits per target")
for b in bands:
    save_visit_map(cat, f"n_images_{b}", f"sky_map_visits_{b}.png", f"LaStBeRu × DP2 — individual visits in band {b}")
print(f"Maps saved in: {out}")


## Cutouts

Los cutouts se generan con `generate_reference_catalog_cutouts` usando los coadds existentes y solo para los grades configurados (A y B). Los FITS se guardan si `save_cutout_fits=true` en la configuración.

In [ ]:
if RUN_CUTOUTS:
    if "cutout_selected" not in catalog:
        raise RuntimeError("Ejecutá primero la etapa de coadds.")
    catalog = generate_reference_catalog_cutouts(config, catalog, reuse_existing=not REGENERATE_CUTOUTS, fits_output_dir=FITS_OUTPUT_DIR, cutout_output_dir=CUTOUT_OUTPUT_DIR / "cutouts")
    catalog.to_csv(out / "reference_catalog_with_cutouts.csv", index=False)
    print(f"Cutouts and final catalog saved in: {out}")
    if "cutout_status" in catalog:
        display(catalog["cutout_status"].value_counts(dropna=False).rename_axis("status").to_frame("targets"))


# Imagen RGB a partir de los FITS generados; se usan z/r/g cuando están disponibles
# y se omiten targets sin al menos tres bandas válidas.
from astropy.io import fits

def make_rgb_images(frame):
    rgb_dir = RGB_OUTPUT_DIR
    rgb_dir.mkdir(exist_ok=True)
    for row in frame.loc[frame["cutout_status"].isin(["generated", "reused"])].itertuples(index=False):
        arrays = {}
        for band in bands:
            path = FITS_OUTPUT_DIR / f"{row.target_id}_{band}.fits"
            if path.exists():
                arrays[band] = fits.getdata(path).astype(float)
        rgb = make_science_rgb(arrays)
        plt.imsave(rgb_dir / f"{row.target_id}_rgb.png", rgb)
    return rgb_dir

if RUN_CUTOUTS:
    print(f"RGB images saved in: {make_rgb_images(catalog)}")
